# Lesson 3: Building an Agent Reasoning Loop

Build a multi-step research agent over a PDF using Gemini function calling and LlamaIndex's `FunctionAgent` workflow API.

## Setup

In [1]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
import nest_asyncio

nest_asyncio.apply()

## Load the data

Ensure `metagpt.pdf` is in this folder:

```bash
curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf
```

## Setup the Query Tools

In [3]:
from utils import get_doc_tools

vector_tool, summary_tool = get_doc_tools("metagpt.pdf", "metagpt")

## Setup Function Calling Agent

Modern LlamaIndex uses `FunctionAgent` (the older `FunctionCallingAgentWorker` / `AgentRunner` APIs were removed).

In [4]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context
from helper import DEFAULT_LLM_MODEL

llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)

agent = FunctionAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    system_prompt=(
        "You are a helpful research assistant over the MetaGPT paper. "
        "Always use the provided tools to answer questions."
    ),
    verbose=True,
)

# Context holds conversation memory across turns
ctx = Context(agent)

2026-07-14 11:16:42,199 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"


In [5]:
ctx

In [6]:
response = await agent.run(
    "Tell me about the agent roles in MetaGPT, "
    "and then how they communicate with each other.",
    ctx=ctx,
)
print(str(response))

2026-07-14 11:17:02,128 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the agent roles in MetaGPT, and then...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-07-14 11:17:02,128 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-07-14 11:17:02,382 - INFO - [init_run:0] complete with AgentInput
2026-07-14 11:17:02,382 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT, and then how they communica...
2026-07-14 11:17:02,382 - INFO - [setup_agent:0] started from AgentInput
2026-07-14 11:17:02,383 - INFO - [setup_agent:0] complete with AgentSetup
2026-07-14 11:17:02,383 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a helpful research assistant over the MetaGPT paper.

In MetaGPT, the framework is designed as a virtual software company with specific agent roles and a structured communication system to ensure efficiency and clarity.

### Agent Roles
MetaGPT defines five primary agent roles, each with a distinct profile including goals, constraints, and specialized skills:
*   **Product Manager:** Responsible for analyzing requirements and creating product documentation. They can utilize tools like web search.
*   **Architect:** Designs the system architecture, producing deliverables such as sequence flow diagrams and system interface designs.
*   **Project Manager:** Manages the project timeline and task allocation.
*   **Engineer:** Responsible for writing the actual code based on the architectural designs. They have the ability to execute code.
*   **QA Engineer:** Focuses on testing and ensuring the quality of the software.

### Communication Mechanism
Rather than relying on open-ended dialogue, which can be inefficient, MetaGPT uses a structured c

In [9]:
# Inspect tool call sources when available
if getattr(response, "tool_calls", None):
    print(response.tool_calls)
else:
    print(response)

[ToolCallResult(tool_name='vector_tool_metagpt', tool_kwargs={'query': 'agent roles in MetaGPT and how they communicate'}, tool_id='vector_tool_metagpt', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='MetaGPT defines five specific agent roles within its software company framework: Product Manager, Architect, Project Manager, Engineer, and QA Engineer. Each role is assigned a profile containing a name, goal, constraints, specific context, and specialized skills; for example, the Product Manager can utilize web search tools, while the Engineer has the ability to execute code.\n\nCommunication among these agents is handled through the following mechanisms:\n\n*   **Structured Communication:** Instead of using dialogue, agents communicate via structured outputs such as documents and diagrams. For instance, the Architect produces a sequence flow diagram and system interface design to serve as deliverables for Engineers.\n*   **Publish-Subscribe Mechanism:** To avoid the i

In [10]:
response = await agent.run(
    "Tell me about the evaluation datasets used.",
    ctx=ctx,
)
print(str(response))

2026-07-14 11:19:46,854 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the evaluation datasets used.', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-07-14 11:19:46,855 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-07-14 11:19:47,019 - INFO - [init_run:0] complete with AgentInput
2026-07-14 11:19:47,019 - INFO - [tick] add: AgentInput(input=[5 items], current_agent_name='Agent')
2026-07-14 11:19:47,019 - INFO - [setup_agent:0] started from AgentInput
2026-07-14 11:19:47,020 - INFO - [setup_agent:0] complete with AgentSetup
2026-07-14 11:19:47,021 - INFO - [tick] add: AgentSetup(input=[6 items], current_agent_name='Agent')
2026-07-14 11:19:47,021 - INFO - [run_agent_step:0] started from AgentSetup
2026-07-14 11:20:03,430 - INFO - [run_agent_step:0] complete with AgentOutput
2026-07-14 11:20:03,430 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwarg

MetaGPT was evaluated using the **SoftwareDev dataset**, which consists of 70 diverse software development tasks.


In [ ]:
response = await agent.run(
    "Tell me the results over one of the above datasets.",
    ctx=ctx,
)
print(str(response))

## Lower-Level: Debuggability and Control

Instead of the old `create_task` / `run_step` API, stream workflow events to observe each tool call as it happens.

In [14]:
from llama_index.core.agent.workflow import AgentStream, ToolCall, ToolCallResult

agent = FunctionAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    system_prompt=(
        "You are a helpful research assistant over the MetaGPT paper. "
        "Always use the provided tools to answer questions."
    ),
    verbose=False,
)
ctx = Context(agent)

In [12]:
handler = agent.run(
    "Tell me about the agent roles in MetaGPT, "
    "and then how they communicate with each other.",
    ctx=ctx,
)

async for event in handler.stream_events():
    if isinstance(event, ToolCall):
        print(f"[tool call] {event.tool_name}({event.tool_kwargs})")
    elif isinstance(event, ToolCallResult):
        print(f"[tool result] {event.tool_name} -> {str(event.tool_output)[:300]}...")
    elif isinstance(event, AgentStream):
        # token stream (optional)
        pass

response = await handler
print("\nFinal response:\n", str(response))

2026-07-14 11:21:13,243 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the agent roles in MetaGPT, and then...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-07-14 11:21:13,244 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-07-14 11:21:13,524 - INFO - [init_run:0] complete with AgentInput
2026-07-14 11:21:13,525 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT, and then how they communica...
2026-07-14 11:21:13,525 - INFO - [setup_agent:0] started from AgentInput
2026-07-14 11:21:13,525 - INFO - [setup_agent:0] complete with AgentSetup
2026-07-14 11:21:13,526 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a helpful research assistant over the MetaGPT paper.

[tool call] vector_tool_metagpt({'query': 'agent roles in MetaGPT and how they communicate'})


2026-07-14 11:21:36,958 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 11:21:36,964 - INFO - [call_tool:0] complete with ToolCallResult
2026-07-14 11:21:36,966 - INFO - [tick] add: ToolCallResult(tool_name='vector_tool_metagpt', tool_kwargs={'query': 'agent roles in MetaGPT and how they communicate'}, tool_id='vector_tool_metagpt', tool_output=ToolOutput(blocks=[TextBlock(blo...
2026-07-14 11:21:36,967 - INFO - [aggregate_tool_results:0] started from ToolCallResult
2026-07-14 11:21:37,090 - INFO - [aggregate_tool_results:0] complete with AgentInput
2026-07-14 11:21:37,091 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT, and then how they communica...
2026-07-14 11:21:37,091 - INFO - [setup_agent:0] started from AgentInput
2026-07-14 11:21:37,092 - INF

[tool result] vector_tool_metagpt -> MetaGPT defines five specific agent roles within its software company framework: Product Manager, Architect, Project Manager, Engineer, and QA Engineer. Each role is assigned a profile consisting of a name, goal, constraints, specific context, and specialized skills; for example, the Product Manager...


2026-07-14 11:21:49,567 - INFO - [run_agent_step:0] complete with AgentOutput
2026-07-14 11:21:49,567 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'thought_signatures': [None, None, None, None, None], 'prompt_tokens': 709, 'completion_tokens': 337, ...
2026-07-14 11:21:49,567 - INFO - [parse_agent_output:0] started from AgentOutput
2026-07-14 11:21:49,727 - INFO - [result] StopEvent(result=AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'thought_signatures': [None, None, None, None, None], 'prompt_tokens': 709, 'completi...
2026-07-14 11:21:49,727 - INFO - [parse_agent_output:0] complete with StopEvent



Final response:
 In MetaGPT, the framework is designed as a virtual software company with five specific agent roles, each with its own profile (name, goal, constraints, context, and specialized skills):

### **Agent Roles**
*   **Product Manager:** Responsible for analyzing requirements and creating the Product Requirement Document (PRD).
*   **Architect:** Takes the PRD and designs the system architecture, producing sequence flow diagrams and system interface designs.
*   **Project Manager:** Manages the project timeline and task allocation.
*   **Engineer:** Implements the code based on the architectural designs.
*   **QA Engineer:** Tests the implemented code to ensure it meets the requirements.

### **Communication Mechanism**
Rather than relying on traditional dialogue, which can be inefficient or ambiguous, MetaGPT uses a structured communication system:

*   **Structured Communication:** Agents communicate via standardized deliverables (documents and diagrams) rather than chat.

In [15]:
# Continue the same conversation with extra guidance mid-reasoning
handler = agent.run(
    "What about how agents share information?",
    ctx=ctx,
)
response = await handler
print(str(response))

2026-07-14 11:22:36,148 - INFO - AFC is enabled with max remote calls: 10.
2026-07-14 11:22:51,231 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


In MetaGPT, agents share information through a **publish-subscribe mechanism** utilizing a **global shared message pool**. 

Key details of this process include:
*   **Global Message Pool:** Instead of using one-to-one dialogue or direct communication, agents publish structured messages (such as documents and diagrams) into a shared pool.
*   **Transparent Access:** Other agents can access the information they need directly from this pool, which eliminates the need to inquire with other agents and wait for a response.
*   **Subscription Mechanism:** To avoid information overload, agents use a subscription system to filter and extract only the information relevant to their specific role and interests.
*   **Dependency-Based Activation:** Agents typically only trigger their actions once all of their prerequisite dependencies (the necessary information from the pool) have been met.
